In [14]:
# Cell 1 — Mount Drive and install packages
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PhishGuard_v2'

import os, subprocess
subprocess.run(['pip', 'install',
    'xgboost==2.1.0', 'shap==0.45.0', 'scikit-learn==1.5.0',
    'pandas==2.2.0', 'numpy==1.26.0', 'joblib==1.4.0', '-q'], check=False)

assert os.path.exists(f'{DRIVE_BASE}/features/contract_features.csv'), \
    'contract_features.csv not found — run Notebook 03 first'
assert os.path.exists(f'{DRIVE_BASE}/models/contract_feature_schema.json'), \
    'contract_feature_schema.json not found — run Notebook 03 first'
print('Cell 1 ready.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cell 1 ready.


In [15]:
# Cell 2 — Load features and schema, validate inputs
import pandas as pd
import numpy as np
import json
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import xgboost as xgb
from xgboost import XGBClassifier
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

MODEL_PATH  = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'
SCHEMA_PATH = f'{DRIVE_BASE}/models/contract_feature_schema.json'
EVAL_DIR    = f'{DRIVE_BASE}/evaluation'

with open(SCHEMA_PATH) as f:
    FEATURE_COLS = json.load(f)

df = pd.read_csv(f'{DRIVE_BASE}/features/contract_features.csv')

TOTAL  = len(df)
PHISH  = (df['label'] == 1).sum()
BENIGN = (df['label'] == 0).sum()

assert df.shape[1] == 23,              f'Wrong column count: {df.shape[1]}'
assert PHISH == BENIGN,                f'Imbalanced: phishing={PHISH} benign={BENIGN}'
assert TOTAL >= 1000,                  f'Too few rows: {TOTAL}'
assert df.isnull().sum().sum() == 0,   'Nulls found'
assert len(FEATURE_COLS) == 21,        f'Schema length wrong: {len(FEATURE_COLS)}'
assert all(c in df.columns for c in FEATURE_COLS), 'Missing feature columns'

X = df[FEATURE_COLS].values
y = df['label'].values

assert X.shape[1] == len(FEATURE_COLS), \
    f'Feature count mismatch: {X.shape[1]} vs {len(FEATURE_COLS)}'

print(f'Loaded: {df.shape}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Rows: {TOTAL} | Phishing: {PHISH} | Benign: {BENIGN}')
print(f'Training features:')
for f in FEATURE_COLS:
    print(f'  {f}')
print('Cell 2 validation passed.')


Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7fc0169ee340>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error


Loaded: (1300, 23)
Features: 21
Rows: 1300 | Phishing: 650 | Benign: 650
Training features:
  is_verified
  bytecode_size
  abi_function_count
  external_public_function_count
  approval_related_function_flag
  permit_related_function_flag
  setApprovalForAll_flag
  opcode_freq_CALL
  opcode_freq_DELEGATECALL
  opcode_freq_SELFDESTRUCT
  opcode_freq_SSTORE
  opcode_freq_JUMPI
  external_call_sites_count
  has_create2
  proxy_pattern_detected
  approval_then_external_call_pattern
  approval_then_state_mutation_pattern
  control_flow_complexity_score
  slither_warning_count_total
  slither_low_level_call_count
  slither_access_control_issues_count
Cell 2 validation passed.


In [17]:
# Cell 3 — Train/test split
all_idx = np.arange(len(y))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.30, stratify=y, random_state=42)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'Train: {X_train.shape} | phishing: {y_train.sum()} | benign: {(y_train==0).sum()}')
print(f'Test:  {X_test.shape}  | phishing: {y_test.sum()}  | benign: {(y_test==0).sum()}')
assert len(test_idx)  == 390,  f'Test size wrong: {len(test_idx)}'
assert len(train_idx) == 910, f'Train size wrong: {len(train_idx)}'

os.makedirs(f'{DRIVE_BASE}/models', exist_ok=True)
np.save(f'{DRIVE_BASE}/models/contract_test_indices.npy', test_idx)
print('Test indices saved.')


Train: (910, 21) | phishing: 455 | benign: 455
Test:  (390, 21)  | phishing: 195  | benign: 195
Test indices saved.


In [18]:
# Cell 4 — XGBoost training with 5-fold cross-validation
scale_pos_weight = float((y_train == 0).sum()) / float((y_train == 1).sum())
print(f'scale_pos_weight: {scale_pos_weight:.4f}')

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    tree_method='hist',
    device='cpu',
    random_state=42
)

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train,
                         cv=cv, scoring='average_precision', n_jobs=1)
print(f'CV PR-AUC per fold: {np.round(scores, 4)}')
print(f'Mean CV PR-AUC: {scores.mean():.4f} ± {scores.std():.4f}')

if scores.mean() < 0.75:
    print('WARNING: Below 0.75 target. Debug before proceeding.')
else:
    print('CV target met. Proceed to Cell 5.')


scale_pos_weight: 1.0000
CV PR-AUC per fold: [0.9529 0.9618 0.9529 0.9584 0.9749]
Mean CV PR-AUC: 0.9602 ± 0.0081
CV target met. Proceed to Cell 5.


In [20]:
# Fit final model on full training set
model.fit(X_train, y_train)
print('Model fitted on training set.')

Model fitted on training set.


In [21]:
# Cell 5 — Evaluation on held-out test set
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_prob)
cm   = confusion_matrix(y_test, y_pred)

print('=== TEST SET EVALUATION ===')
print(f'  Accuracy:  {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1:        {f1:.4f}')
print(f'  ROC-AUC:   {auc:.4f}')
print(f'\nConfusion Matrix:\n{cm}')
print(f'\n{classification_report(y_test, y_pred, target_names=["benign","phishing"])}')

assert f1  >= 0.70, f'F1 too low: {f1:.4f}'
assert auc >= 0.75, f'AUC too low: {auc:.4f}'
print('Minimum performance thresholds passed.')



=== TEST SET EVALUATION ===
  Accuracy:  0.8872
  Precision: 0.9081
  Recall:    0.8615
  F1:        0.8842
  ROC-AUC:   0.9384

Confusion Matrix:
[[178  17]
 [ 27 168]]

              precision    recall  f1-score   support

      benign       0.87      0.91      0.89       195
    phishing       0.91      0.86      0.88       195

    accuracy                           0.89       390
   macro avg       0.89      0.89      0.89       390
weighted avg       0.89      0.89      0.89       390

Minimum performance thresholds passed.


In [23]:
# Cell 6 — SHAP feature importance
os.makedirs(EVAL_DIR, exist_ok=True)

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary bar plot — mean |SHAP| per feature
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=FEATURE_COLS,
                  plot_type='bar',
                  show=False)
plt.tight_layout()
bar_path = f'{EVAL_DIR}/contract_shap_bar.png'
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {bar_path}')

# Beeswarm plot
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test,
                  feature_names=FEATURE_COLS,
                  show=False)
plt.tight_layout()
bee_path = f'{EVAL_DIR}/contract_shap_beeswarm.png'
plt.savefig(bee_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {bee_path}')

# Top features by mean |SHAP|
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx       = np.argsort(mean_abs_shap)[::-1]
print('\nFeature importance (mean |SHAP|):')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank}. {FEATURE_COLS[idx]:<45}  {mean_abs_shap[idx]:.5f}')

print('Cell 6 SHAP complete.')


Saved: /content/drive/MyDrive/PhishGuard_v2/evaluation/contract_shap_bar.png
Saved: /content/drive/MyDrive/PhishGuard_v2/evaluation/contract_shap_beeswarm.png

Feature importance (mean |SHAP|):
  1. bytecode_size                                  1.18019
  2. control_flow_complexity_score                  0.71997
  3. is_verified                                    0.69709
  4. opcode_freq_SSTORE                             0.64652
  5. opcode_freq_CALL                               0.63824
  6. abi_function_count                             0.54216
  7. opcode_freq_JUMPI                              0.49159
  8. external_public_function_count                 0.26483
  9. external_call_sites_count                      0.18339
  10. opcode_freq_DELEGATECALL                       0.15989
  11. slither_warning_count_total                    0.07188
  12. approval_then_state_mutation_pattern           0.04977
  13. approval_related_function_flag                 0.04799
  14. slither_low_leve

In [24]:
# Cell 7 — Save model and verify
MODEL_PATH = f'{DRIVE_BASE}/models/contract_xgboost_v1.pkl'
joblib.dump(model, MODEL_PATH)
print(f'Saved: {MODEL_PATH}')

loaded     = joblib.load(MODEL_PATH)
test_preds = loaded.predict_proba(X_test[:5])[:, 1]
print(f'Verify predictions: {np.round(test_preds, 4)}')
print('Model save verified.')
print('Notebook 05 complete.')


Saved: /content/drive/MyDrive/PhishGuard_v2/models/contract_xgboost_v1.pkl
Verify predictions: [0.859  0.0248 0.0095 0.8368 0.9969]
Model save verified.
Notebook 05 complete.
